### Week 4 Graph Analysis

这个 notebook 用来检查 concept co-occurrence graph 是否太稀、太密，或者被泛词/噪声污染。  
主要观察节点数、边数、平均度、connected components、top degree / weighted degree / PageRank concepts，以及每个 passage 的 concept 数量分布。

#### 1. 配置路径

In [1]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

GRAPH_PATH = PROJECT_ROOT / "data/processed/concept_graph.pkl"
PASSAGE_CONCEPTS_PATH = PROJECT_ROOT / "data/processed/concepts/passage_concepts.jsonl"
OUTPUT_PATH = PROJECT_ROOT / "reports/results/week4_graph_stats.csv"

GRAPH_PATH, PASSAGE_CONCEPTS_PATH, OUTPUT_PATH

(PosixPath('/home/haris/hotpot-evidence-retrieval/data/processed/concept_graph.pkl'),
 PosixPath('/home/haris/hotpot-evidence-retrieval/data/processed/concepts/passage_concepts.jsonl'),
 PosixPath('/home/haris/hotpot-evidence-retrieval/reports/results/week4_graph_stats.csv'))

#### 2. 运行图统计

In [2]:
from graph.analyze_graph import (
    GRAPH_STATS_COLUMNS,
    build_graph_stats_rows,
)
from utils.jsonl_io import load_graph, load_jsonl, write_csv

graph = load_graph(GRAPH_PATH)
passage_concepts = load_jsonl(PASSAGE_CONCEPTS_PATH)
rows = build_graph_stats_rows(graph, passage_concepts, top_n=20)
write_csv(rows, OUTPUT_PATH, fieldnames=GRAPH_STATS_COLUMNS)

print(f"Wrote {len(rows)} rows to {OUTPUT_PATH}")

Wrote 102 rows to /home/haris/hotpot-evidence-retrieval/reports/results/week4_graph_stats.csv


#### 3. 总体图结构

In [3]:
def read_stats(path):
    with path.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def show_rows(selected_rows, columns=("section", "rank", "metric", "concept", "value", "extra")):
    widths = {column: len(column) for column in columns}
    for row in selected_rows:
        for column in columns:
            widths[column] = max(widths[column], len(str(row[column])))

    header = " | ".join(column.ljust(widths[column]) for column in columns)
    divider = " | ".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in selected_rows:
        print(" | ".join(str(row[column]).ljust(widths[column]) for column in columns))


stats_rows = read_stats(OUTPUT_PATH)
summary_rows = [row for row in stats_rows if row["section"] == "graph_summary"]
show_rows(summary_rows, columns=("metric", "value", "extra"))

metric                 | value     | extra                               
---------------------- | --------- | ------------------------------------
num_nodes              | 17163     | |V|                                 
num_edges              | 113966    | |E|                                 
average_degree         | 13.280429 | 2*|E|/|V|                           
connected_components   | 204       | weak components if graph is directed
largest_component_size | 16556     | nodes in largest component          


##### 观察

- **主图占绝对多数，碎片分量规模很小**：最大 connected component 包含 96.46% (16556个) 的节点，其余 203 个分量都很小，最大也只有 10 个节点，说明图没有明显碎裂。
- **图整体可用，但存在泛词 hub 风险**：平均度为 13.28，且最大分量占比很高，说明图不是过度稀疏；但大型连通核心明显，需要警惕泛词 hub 把不相关 concepts 连接在一起。

#### 4. Top Degree Concepts

In [4]:
top_degree_rows = [row for row in stats_rows if row["section"] == "top_degree_concepts"]
show_rows(top_degree_rows[:20], columns=("rank", "concept", "value"))

rank | concept       | value
---- | ------------- | -----
1    | american      | 1210 
2    | united states | 859  
3    | english       | 554  
4    | second        | 541  
5    | film          | 501  
6    | album         | 409  
7    | song          | 402  
8    | british       | 390  
9    | italian       | 350  
10   | time          | 344  
11   | university    | 322  
12   | series        | 307  
13   | year          | 287  
14   | u s           | 279  
15   | australia     | 274  
16   | season        | 274  
17   | september     | 273  
18   | australian    | 271  
19   | members       | 263  
20   | october       | 261  


##### 观察

- **高 degree 节点受泛词主导**：例如 `american`、`united states`、`english`、`second`、`film`、`album`、`song`、`british`、`year`、月份词等范词或高频词，这更多反应语料中泛词的高出现频率，及当前 concept 抽取规则对泛词的保留倾向，并不代表这些词能有效区分相关 evidence。
- **存在少数强 hub 压缩语义距离**：`american` 连接 1,210 个 concepts，`united states` 连接 859 个，虽然还没有覆盖整个图，但已经足以把许多主题无关的节点拉得更近。

#### 5. Top Weighted Degree Concepts

In [5]:
top_weighted_degree_rows = [row for row in stats_rows if row["section"] == "top_weighted_degree_concepts"]
show_rows(top_weighted_degree_rows[:20], columns=("rank", "concept", "value"))

rank | concept       | value
---- | ------------- | -----
1    | american      | 1463 
2    | united states | 1002 
3    | english       | 642  
4    | second        | 620  
5    | song          | 533  
6    | film          | 532  
7    | album         | 468  
8    | university    | 418  
9    | british       | 410  
10   | italian       | 388  
11   | time          | 368  
12   | commonwealth  | 349  
13   | series        | 339  
14   | australia     | 329  
15   | members       | 327  
16   | season        | 323  
17   | year          | 313  
18   | australian    | 307  
19   | u s           | 300  
20   | september     | 295  


##### 观察

- **Weighted degree 仍由泛词 hub 主导**: 例如 `american`、`united states`、`english`、`second`、`song`、`film`、`album`，这些更像噪声 hub，而不是稳定有用的语义桥接节点。
- **高频 hub 不只是连接多，也反复共现**：Weighted degree 和普通 degree 的 top concepts 差异不大，只是局部排序变化，这说明这些 hub 不只是连接对象多，而且在多个 passages 中反复共现。

#### 6. Top PageRank Concepts

In [6]:
top_pagerank_rows = [row for row in stats_rows if row["section"] == "top_pagerank_concepts"]
show_rows(top_pagerank_rows[:20], columns=("rank", "concept", "value"))

rank | concept       | value         
---- | ------------- | --------------
1    | american      | 0.005494588862
2    | united states | 0.003398277973
3    | film          | 0.002223824729
4    | second        | 0.002222939441
5    | english       | 0.002151465685
6    | song          | 0.002033114610
7    | album         | 0.001759739705
8    | italian       | 0.001432571207
9    | university    | 0.001426896047
10   | series        | 0.001409279314
11   | time          | 0.001357268774
12   | british       | 0.001355245391
13   | season        | 0.001144218875
14   | u s           | 0.001082850104
15   | september     | 0.001066648642
16   | members       | 0.001043418639
17   | year          | 0.001014504825
18   | 2010          | 0.001013023269
19   | october       | 0.000990499278
20   | australia     | 0.000964563749


##### 观察

- **PageRank 仍被泛词主导**：PageRank top concepts 与 degree top concepts 差异不大，仍集中在 `american`、`united states`、`film`、`second`、`english`、`song`、`album`、`series`、`season` 等宽泛词，说明 PageRank 没有明显筛出更有语义价值的节点。
- **使用 PageRank 前需要过滤泛词**： 这些泛词不仅自身 degree 高，而且连接到的邻居也位于主分量的重要位置；因此如果后续要把 PageRank 作为 graph feature，需要先过滤或降权这类泛词。

#### 7. Passage Concept 数量分布

In [7]:
distribution_rows = [row for row in stats_rows if row["section"] == "passage_concept_count_distribution"]
show_rows(distribution_rows, columns=("value", "extra"))

value | extra                              
----- | -----------------------------------
13    | concept_count=0; fraction=0.003182 
84    | concept_count=1; fraction=0.020563 
178   | concept_count=2; fraction=0.043574 
346   | concept_count=3; fraction=0.084700 
460   | concept_count=4; fraction=0.112607 
574   | concept_count=5; fraction=0.140514 
514   | concept_count=6; fraction=0.125826 
440   | concept_count=7; fraction=0.107711 
353   | concept_count=8; fraction=0.086414 
280   | concept_count=9; fraction=0.068543 
215   | concept_count=10; fraction=0.052632
167   | concept_count=11; fraction=0.040881
139   | concept_count=12; fraction=0.034027
88    | concept_count=13; fraction=0.021542
69    | concept_count=14; fraction=0.016891
42    | concept_count=15; fraction=0.010282
32    | concept_count=16; fraction=0.007834
22    | concept_count=17; fraction=0.005386
10    | concept_count=18; fraction=0.002448
9     | concept_count=19; fraction=0.002203
18    | concept_count=20; fracti

##### 观察

- **单个 passage 的 concept 数量整体可控**：平均 7.00 个，中位数 6 个，90 分位数 12 个，说明多数 passages 没有抽出过多 concepts。
- **存在少量长尾异常样本**：少数 passages 的 concept 数量明显偏高，32 个超过 20 个 concepts，9 个超过 30 个，最高达到 56 个，后续应回看这些样本，并考虑设置 concept 上限或加强过滤。

#### Conclusion

- 总体来看，当前 concept co-occurrence graph **结构上是可用的，但直接用于 graph-based retrieval feature 仍有明显噪声风险**。图没有过度稀疏或严重碎裂，大多数 concepts 被连接进同一个主图，说明它能提供较完整的连接结构。

- 主要问题是中心性指标被泛词 hub 明显主导。Top degree、weighted degree 和 PageRank 都集中在 american、united states、english、film、song、album 等宽泛高频 concepts 上。如果直接使用这些中心性或邻接关系，可能会把不相关 evidence 拉近，降低 graph signal 的有效性。

- Passage-level concept 数量整体比较健康，多数 passages 抽出的 concepts 数量可控。但仍存在少量长尾异常样本，需要后续检查，并考虑设置每个 passage 的 concept 上限或加强 concept 过滤。

- 因此，下一步不应直接把原始 graph feature 全量接入 retrieval，而应先做 concept 过滤和 hub 控制，再比较过滤前后的 retrieval 指标。当前 graph 更适合作为清洗后的辅助信号，而不是直接作为主排序信号。